In [0]:
%run ./00_config

In [0]:
raw = spark.table(f"{SRC}.facilities")

raw.printSchema()
display(raw.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw = spark.table(f"{SRC}.facilities")
raw_count = raw.count()

print("raw number：", raw_count)

display(raw.select(
    "unique_id", "name", "address_city",
    "procedure", "equipment", "capability",
    "source_urls", "coordinates"
).limit(10))

In [0]:
def clean_text(column_name):
    value = F.trim(F.regexp_replace(F.col(column_name), r"\s+", " "))
    return F.when(
        value.isNull()
        | F.lower(value).isin("", "null", "none", "n/a", "na"),
        F.lit(None).cast("string")
    ).otherwise(value)

clean = raw

text_fields = {
    "unique_id": "facility_id",
    "name": "name_clean",
    "address_city": "city_clean",
    "address_stateOrRegion": "state_clean",
    "address_zipOrPostcode": "postcode_clean",
    "facilityTypeId": "facility_type_clean",
    "operatorTypeId": "operator_type_clean",
    "cluster_id": "cluster_id_clean",
}

for source, target in text_fields.items():
    clean = clean.withColumn(target, clean_text(source))

for source, target in [
    ("name_clean", "name_key"),
    ("city_clean", "city_key"),
    ("state_clean", "state_key"),
]:
    clean = clean.withColumn(target, F.lower(F.col(source)))

clean = (
    clean
    .withColumn("flag_missing_id", F.col("facility_id").isNull())
    .withColumn("flag_missing_name", F.col("name_clean").isNull())
    .withColumn("flag_missing_postcode", F.col("postcode_clean").isNull())
    .withColumn(
        "flag_invalid_postcode",
        F.col("postcode_clean").isNotNull()
        & ~F.col("postcode_clean").rlike(r"^[1-9][0-9]{5}$")
    )
)

In [0]:
for field in ["procedure", "equipment", "capability"]:
    text = F.trim(F.col(field))
    missing = (
        text.isNull()
        | F.lower(text).isin("", "null", "none", "n/a", "na")
    )

    clean = clean.withColumn(
        f"{field}_items",
        F.when(
            ~missing & text.startswith("["),
            F.from_json(F.col(field), "array<string>")
        )
    )

    clean = (
        clean
        .withColumn(f"flag_{field}_missing", missing)
        .withColumn(
            f"flag_{field}_parse_failed",
            ~missing & F.col(f"{field}_items").isNull()
        )
    )

In [0]:
lat = F.col("latitude")
lon = F.col("longitude")

missing_coord = (
    lat.isNull() | lon.isNull()
    | F.isnan(lat) | F.isnan(lon)
)

invalid_coord = (
    ~missing_coord
    & (~lat.between(-90, 90) | ~lon.between(-180, 180))
)

outside_bbox = (
    ~missing_coord
    & ~invalid_coord
    & (~lat.between(6, 38) | ~lon.between(68, 98))
)

clean = (
    clean
    .withColumn("flag_missing_coordinates", missing_coord)
    .withColumn("flag_invalid_coordinates", invalid_coord)
    .withColumn("flag_outside_india_bbox", outside_bbox)
)

In [0]:
def duplicate_flag(keys):
    complete = F.lit(True)
    for key in keys:
        complete = complete & F.col(key).isNotNull()

    count = F.count(F.lit(1)).over(Window.partitionBy(*keys))
    return complete & (count > 1)

clean = (
    clean
    .withColumn(
        "flag_duplicate_id",
        duplicate_flag(["facility_id"])
    )
    .withColumn(
        "flag_duplicate_name_city",
        duplicate_flag(["name_key", "city_key"])
    )
    .withColumn(
        "flag_duplicate_cluster",
        duplicate_flag(["cluster_id_clean"])
    )
)

In [0]:
clean_count = clean.count()
assert clean_count == raw_count, "data number error"

flag_cols = [c for c in clean.columns if c.startswith("flag_")]

display(clean.agg(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).cast("int")).alias(c)
        for c in flag_cols
    ]
))

display(clean.select(
    "facility_id",
    "name_clean",
    "city_clean",
    "state_clean",
    "postcode_clean",
    "procedure_items",
    "equipment_items",
    "capability_items",
    *flag_cols
).limit(20))

In [0]:
OUTPUT_TABLE = f"{OUT}.ann_facilities_clean_v1"

clean.createOrReplaceTempView("ann_clean_to_save")

spark.sql(f"""
    CREATE TABLE {OUTPUT_TABLE}
    USING DELTA
    AS SELECT * FROM ann_clean_to_save
""")

print("已保存：", OUTPUT_TABLE)

display(spark.table(OUTPUT_TABLE).limit(5))